# 1. Импорт библиотек и настройки

In [1]:
import pandas as pd

In [2]:
from pprint import pprint

In [3]:
pd.set_option('display.float_format', '{:.4f}'.format)

# 2. Загрузка констант и динамики

In [4]:
const = { 'Привлекаемые_средства': 380_000_000_000,
            'Ставка_купона_ОФЗ_ИН_л': 0.025,
            'Ставка_купона_ОФЗ_ПД': 0.1374,
            'Номинал_ОФЗ_ИН': 10_000,
            'Номинал_ОФЗ_ПД': 1000,
            'Количество_человек': 2_000_000,
            'НДФЛ': 0.13
        }
pprint(const)

{'Количество_человек': 2000000,
 'НДФЛ': 0.13,
 'Номинал_ОФЗ_ИН': 10000,
 'Номинал_ОФЗ_ПД': 1000,
 'Привлекаемые_средства': 380000000000,
 'Ставка_купона_ОФЗ_ИН_л': 0.025,
 'Ставка_купона_ОФЗ_ПД': 0.1374}


In [5]:
dynamic = pd.read_excel(
    "Модель по ОФЗ ИН (в) переделыш.xlsx",
    sheet_name="Входные данные",    # имя листа
    usecols="A:C",                 #  параметр, который говорит: «бери только эти столбцы»
    skiprows=0                     # сколько строк пропустить в начале файла (при чтении)
)
dynamic.columns = ['Год', 'Ставка инфляции', 'Ставка депозита']
dynamic = dynamic.dropna(subset=['Год'])
dynamic['Год'] =  dynamic['Год'].astype(int)
dynamic


,Год,Ставка инфляции,Ставка депозита
0,2026,0.0560,0.1306
1,2027,0.0400,0.0800
2,2028,0.0400,0.0690


# 3. ОФЗ ИН (л)

In [6]:
ofz = dynamic.copy()
ofz

,Год,Ставка инфляции,Ставка депозита
0,2026,0.0560,0.1306
1,2027,0.0400,0.0800
2,2028,0.0400,0.0690


In [7]:
ofz['Привлекаемые средства'] = const['Привлекаемые_средства']
ofz['Количество человек'] = const['Количество_человек']
ofz['Ставка купона'] = const['Ставка_купона_ОФЗ_ИН_л']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона
0,2026,0.0560,0.1306,380000000000,2000000,0.0250
1,2027,0.0400,0.0800,380000000000,2000000,0.0250
2,2028,0.0400,0.0690,380000000000,2000000,0.0250


In [8]:
ofz['На руках у человека, руб'] = ofz['Привлекаемые средства'] / ofz['Количество человек']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб"
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000


In [9]:
ofz['Облигаций штук'] = ofz['На руках у человека, руб'] / const ['Номинал_ОФЗ_ИН']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000


In [10]:
ofz['Инфляционный множитель']= (1 + dynamic['Ставка инфляции']).cumprod()
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422


In [11]:
ofz['Номинал после индексации'] = const['Номинал_ОФЗ_ИН']*ofz['Инфляционный множитель']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560,10560.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982,10982.4000
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422,11421.6960


In [12]:
ofz['Номинал на начало'] = float(const['Номинал_ОФЗ_ИН'])
ofz.loc[ofz.index > 0, 'Номинал на начало'] = ofz['Номинал после индексации'].shift(1).fillna(const['Номинал_ОФЗ_ИН'])
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560,10560.0000,10000.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982,10982.4000,10560.0000
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422,11421.6960,10982.4000


In [13]:
ofz['Индексация номинала'] = ofz['Номинал на начало'] * ofz['Ставка инфляции']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560,10560.0000,10000.0000,560.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982,10982.4000,10560.0000,422.4000
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422,11421.6960,10982.4000,439.2960


In [14]:
ofz['Купон, руб'] = ofz['Номинал после индексации'] * ofz['Ставка купона']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб"
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560,10560.0000,10000.0000,560.0000,264.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982,10982.4000,10560.0000,422.4000,274.5600
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422,11421.6960,10982.4000,439.2960,285.5424


In [15]:
ofz['Доход без вычета'] = ofz['Купон, руб'] * ofz['Облигаций штук']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560,10560.0000,10000.0000,560.0000,264.0000,5016.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982,10982.4000,10560.0000,422.4000,274.5600,5216.6400
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422,11421.6960,10982.4000,439.2960,285.5424,5425.3056


In [16]:
ofz ['Налоговый вычет, руб']= ofz['На руках у человека, руб'] * const['НДФЛ']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета,"Налоговый вычет, руб"
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560,10560.0000,10000.0000,560.0000,264.0000,5016.0000,24700.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982,10982.4000,10560.0000,422.4000,274.5600,5216.6400,24700.0000
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422,11421.6960,10982.4000,439.2960,285.5424,5425.3056,24700.0000


In [17]:
ofz['Доход с вычетом'] = ofz['Налоговый вычет, руб'] + ofz['Доход без вычета']
ofz

,Год,Ставка инфляции,Ставка депозита,Привлекаемые средства,Количество человек,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал после индексации,Номинал на начало,Индексация номинала,"Купон, руб",Доход без вычета,"Налоговый вычет, руб",Доход с вычетом
0,2026,0.0560,0.1306,380000000000,2000000,0.0250,190000.0000,19.0000,1.0560,10560.0000,10000.0000,560.0000,264.0000,5016.0000,24700.0000,29716.0000
1,2027,0.0400,0.0800,380000000000,2000000,0.0250,190000.0000,19.0000,1.0982,10982.4000,10560.0000,422.4000,274.5600,5216.6400,24700.0000,29916.6400
2,2028,0.0400,0.0690,380000000000,2000000,0.0250,190000.0000,19.0000,1.1422,11421.6960,10982.4000,439.2960,285.5424,5425.3056,24700.0000,30125.3056


In [18]:
cols = ofz.columns.tolist()
cols

['Год',
 'Ставка инфляции',
 'Ставка депозита',
 'Привлекаемые средства',
 'Количество человек',
 'Ставка купона',
 'На руках у человека, руб',
 'Облигаций штук',
 'Инфляционный множитель',
 'Номинал после индексации',
 'Номинал на начало',
 'Индексация номинала',
 'Купон, руб',
 'Доход без вычета',
 'Налоговый вычет, руб',
 'Доход с вычетом']

In [19]:
df = ofz[['Год',
 'Привлекаемые средства',
 'Количество человек',
 'Ставка инфляции',
 'Ставка купона',
 'На руках у человека, руб',
 'Облигаций штук',
 'Инфляционный множитель',
 'Номинал на начало',
 'Индексация номинала',
 'Номинал после индексации',
 'Купон, руб',
 'Доход без вычета',
 'Налоговый вычет, руб',
 'Доход с вычетом']]
df

,Год,Привлекаемые средства,Количество человек,Ставка инфляции,Ставка купона,"На руках у человека, руб",Облигаций штук,Инфляционный множитель,Номинал на начало,Индексация номинала,Номинал после индексации,"Купон, руб",Доход без вычета,"Налоговый вычет, руб",Доход с вычетом
0,2026,380000000000,2000000,0.0560,0.0250,190000.0000,19.0000,1.0560,10000.0000,560.0000,10560.0000,264.0000,5016.0000,24700.0000,29716.0000
1,2027,380000000000,2000000,0.0400,0.0250,190000.0000,19.0000,1.0982,10560.0000,422.4000,10982.4000,274.5600,5216.6400,24700.0000,29916.6400
2,2028,380000000000,2000000,0.0400,0.0250,190000.0000,19.0000,1.1422,10982.4000,439.2960,11421.6960,285.5424,5425.3056,24700.0000,30125.3056


# 4. ОФЗ ПД

In [20]:
pprint (const)

{'Количество_человек': 2000000,
 'НДФЛ': 0.13,
 'Номинал_ОФЗ_ИН': 10000,
 'Номинал_ОФЗ_ПД': 1000,
 'Привлекаемые_средства': 380000000000,
 'Ставка_купона_ОФЗ_ИН_л': 0.025,
 'Ставка_купона_ОФЗ_ПД': 0.1374}


In [21]:
dynamic

,Год,Ставка инфляции,Ставка депозита
0,2026,0.0560,0.1306
1,2027,0.0400,0.0800
2,2028,0.0400,0.0690


In [22]:
ofz_pd = ofz [['Год']].copy()
ofz_pd['Привлекаемые средства'] = const ["Привлекаемые_средства"]
ofz_pd ["Количество человек"] = const ["Количество_человек"]
ofz_pd ["Ставка купона"] = const ['Ставка_купона_ОФЗ_ПД']
ofz_pd ["На руках у человека"] = ofz [["На руках у человека, руб"]].copy()
ofz_pd ['Облигаций, штук'] = ofz_pd ['На руках у человека'] / const ['Номинал_ОФЗ_ПД']
ofz_pd ['Купон'] = const ['Номинал_ОФЗ_ПД'] * const ['Ставка_купона_ОФЗ_ПД']
ofz_pd ["Доход, руб"] = ofz_pd ["Купон"] * ofz_pd ['Облигаций, штук']
ofz_pd ['НДФЛ'] = ofz_pd ['Доход, руб'] * const ['НДФЛ']
ofz_pd ['Доход после вычета налога'] = ofz_pd['Доход, руб'] - ofz_pd ['НДФЛ']
ofz_pd

,Год,Привлекаемые средства,Количество человек,Ставка купона,На руках у человека,"Облигаций, штук",Купон,"Доход, руб",НДФЛ,Доход после вычета налога
0,2026,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
1,2027,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
2,2028,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200


# 5. Депозит

In [23]:
depozit = ofz[['Год']].copy()
depozit ['Привлекаемые средства'] = const ['Привлекаемые_средства']
depozit ['Количество человек'] = const ['Количество_человек']
depozit ['На руках у человека'] = ofz ['На руках у человека, руб']
depozit ['Ставка депозита'] = dynamic ['Ставка депозита'] 


In [24]:
depozit ['Коэфициент'] = (1 + dynamic ['Ставка депозита'] / 12) **12
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент
0,2026,380000000000,2000000,190000.0000,0.1306,1.1387
1,2027,380000000000,2000000,190000.0000,0.0800,1.0830
2,2028,380000000000,2000000,190000.0000,0.0690,1.0712


In [25]:
depozit ['Накопленный множитель'] = depozit ['Коэфициент'].cumprod()
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель
0,2026,380000000000,2000000,190000.0000,0.1306,1.1387,1.1387
1,2027,380000000000,2000000,190000.0000,0.0800,1.0830,1.2332
2,2028,380000000000,2000000,190000.0000,0.0690,1.0712,1.3211


In [26]:
 initial_amount = depozit['На руках у человека'].iloc[0]

In [27]:
depozit ['Сумма на конец года'] = initial_amount * depozit ['Накопленный множитель']
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года
0,2026,380000000000,2000000,190000.0000,0.1306,1.1387,1.1387,216354.5517
1,2027,380000000000,2000000,190000.0000,0.0800,1.0830,1.2332,234311.8728
2,2028,380000000000,2000000,190000.0000,0.0690,1.0712,1.3211,251000.6177


In [28]:
depozit ['Сумма на начало года'] = initial_amount
depozit.loc [depozit.index > 0, 'Сумма на начало года'] = depozit['Сумма на конец года']. shift (1)
# df_dep['Сумма на начало года '] = df_dep['Сумма на конец года'].shift(1).fillna(df_dep['На руках у человека, руб'].iloc[0])
# df_dep['Сумма начало'] = df_dep['Сумма конец'].shift(1).fillna(initial)
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года,Сумма на начало года
0,2026,380000000000,2000000,190000.0000,0.1306,1.1387,1.1387,216354.5517,190000.0000
1,2027,380000000000,2000000,190000.0000,0.0800,1.0830,1.2332,234311.8728,216354.5517
2,2028,380000000000,2000000,190000.0000,0.0690,1.0712,1.3211,251000.6177,234311.8728


In [29]:
depozit['Проценты']= depozit['Сумма на конец года'] - depozit['Сумма на начало года']
depozit

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на конец года,Сумма на начало года,Проценты
0,2026,380000000000,2000000,190000.0000,0.1306,1.1387,1.1387,216354.5517,190000.0000,26354.5517
1,2027,380000000000,2000000,190000.0000,0.0800,1.0830,1.2332,234311.8728,216354.5517,17957.3211
2,2028,380000000000,2000000,190000.0000,0.0690,1.0712,1.3211,251000.6177,234311.8728,16688.7449


In [30]:
df_dep = depozit [["Год", "Привлекаемые средства", 
                   "Количество человек", 
                   "На руках у человека",
                   "Ставка депозита",
                   "Коэфициент",
                   "Накопленный множитель",
                   "Сумма на начало года",
                   "Сумма на конец года",
                    "Проценты"]]
df_dep          

,Год,Привлекаемые средства,Количество человек,На руках у человека,Ставка депозита,Коэфициент,Накопленный множитель,Сумма на начало года,Сумма на конец года,Проценты
0,2026,380000000000,2000000,190000.0000,0.1306,1.1387,1.1387,190000.0000,216354.5517,26354.5517
1,2027,380000000000,2000000,190000.0000,0.0800,1.0830,1.2332,216354.5517,234311.8728,17957.3211
2,2028,380000000000,2000000,190000.0000,0.0690,1.0712,1.3211,234311.8728,251000.6177,16688.7449


# 6. Доход за период (собрал без merge)

In [31]:
oin_summary = pd.DataFrame({
    'Инструмент': ['ОФЗ ИН (л)'],
    'Номинал': [ofz['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [ofz ['Индексация номинала'].sum()*ofz['Облигаций штук'].iloc[0] +  ofz['Доход без вычета'].sum()],
})
oin_summary ['Итоговая сумма'] = ofz['На руках у человека, руб'].iloc[0] + oin_summary['Доход, до налогов'] + ofz ['Налоговый вычет, руб'].iloc [0] 
oin_summary

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма
0,ОФЗ ИН (л),190000.0000,42670.1696,257370.1696


In [32]:
pd_summary = pd.DataFrame({
    'Инструмент': ['ОФЗ ПД'],
    'Номинал': [ofz ['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [ofz_pd['Доход, руб'].sum()],
    'Итоговая сумма': [ofz ['На руках у человека, руб'].iloc[0] + ofz_pd['Доход, руб'].sum()]})
pd_summary

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма
0,ОФЗ ПД,190000.0000,78318.0000,268318.0000


In [33]:
dep_summary = pd.DataFrame({
    'Инструмент': ['Депозит'],
    'Номинал': [ofz ['На руках у человека, руб'].iloc[0]],
    'Доход, до налогов': [depozit['Проценты'].sum()],
    'Итоговая сумма': [ofz ['На руках у человека, руб'].iloc[0] + depozit['Проценты'].sum()]})
dep_summary

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма
0,Депозит,190000.0000,61000.6177,251000.6177


In [34]:
pohti_konec = pd.concat([oin_summary, pd_summary,dep_summary],ignore_index= True)
pohti_konec

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма
0,ОФЗ ИН (л),190000.0000,42670.1696,257370.1696
1,ОФЗ ПД,190000.0000,78318.0000,268318.0000
2,Депозит,190000.0000,61000.6177,251000.6177


In [35]:
inf_factor = (1 + dynamic['Ставка инфляции']).prod()

In [36]:
pohti_konec['Очистка инфляции'] = pohti_konec['Итоговая сумма']/inf_factor
pohti_konec['Реальный доход'] = pohti_konec['Итоговая сумма'] - pohti_konec['Номинал']
pohti_konec

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма,Очистка инфляции,Реальный доход
0,ОФЗ ИН (л),190000.0000,42670.1696,257370.1696,225334.4596,67370.1696
1,ОФЗ ПД,190000.0000,78318.0000,268318.0000,234919.5776,78318.0000
2,Депозит,190000.0000,61000.6177,251000.6177,219757.7467,61000.6177


In [37]:
pohti_konec.loc[pohti_konec['Инструмент'] == 'ОФЗ ИН (л)', 'Очистка инфляции'] = None
# pohti_konec.loc[pohti_konec['Инструмент'] == 'ОФЗ ИН (л)', 'Реальный доход'] = None
pohti_konec

,Инструмент,Номинал,"Доход, до налогов",Итоговая сумма,Очистка инфляции,Реальный доход
0,ОФЗ ИН (л),190000.0000,42670.1696,257370.1696,NaN,67370.1696
1,ОФЗ ПД,190000.0000,78318.0000,268318.0000,234919.5776,78318.0000
2,Депозит,190000.0000,61000.6177,251000.6177,219757.7467,61000.6177


# Делаем длинную таблицу (получилась широкая, но другая, будем думать дальше)

In [38]:
df_income = ofz[['Год', 'На руках у человека, руб']].copy()
df_income.rename(columns={'На руках у человека, руб': 'Вложения'}, inplace=True)
df_income

,Год,Вложения
0,2026,190000.0000
1,2027,190000.0000
2,2028,190000.0000


In [39]:
df_income['ОФЗ ИН доход'] = ofz['Индексация номинала']* ofz['Облигаций штук'] + ofz['Доход без вычета']
df_income

,Год,Вложения,ОФЗ ИН доход
0,2026,190000.0000,15656.0000
1,2027,190000.0000,13242.2400
2,2028,190000.0000,13771.9296


In [40]:
df_income = df_income.merge(ofz_pd[['Год', 'Доход, руб']], on='Год', how='left')
df_income.rename(columns={'Доход, руб': 'ОФЗ ПД доход'}, inplace=True)
df_income

,Год,Вложения,ОФЗ ИН доход,ОФЗ ПД доход
0,2026,190000.0000,15656.0000,26106.0000
1,2027,190000.0000,13242.2400,26106.0000
2,2028,190000.0000,13771.9296,26106.0000


In [41]:
df_income = df_income.merge(depozit[['Год', 'Проценты']], on='Год', how='left')
df_income.rename(columns={'Проценты': 'Депозит доход'}, inplace=True)
df_income

,Год,Вложения,ОФЗ ИН доход,ОФЗ ПД доход,Депозит доход
0,2026,190000.0000,15656.0000,26106.0000,26354.5517
1,2027,190000.0000,13242.2400,26106.0000,17957.3211
2,2028,190000.0000,13771.9296,26106.0000,16688.7449


In [42]:
# строка итого
total_row = df_income[['ОФЗ ИН доход', 'ОФЗ ПД доход', 'Депозит доход']].sum()
total_row['Год'] = 'Итого'
total_row['Вложения'] = df_income['Вложения'].iloc[0]  # начальные вложения (одинаковы)
df_income = pd.concat([df_income, pd.DataFrame([total_row])], ignore_index=True)
df_income

,Год,Вложения,ОФЗ ИН доход,ОФЗ ПД доход,Депозит доход
0,2026,190000.0000,15656.0000,26106.0000,26354.5517
1,2027,190000.0000,13242.2400,26106.0000,17957.3211
2,2028,190000.0000,13771.9296,26106.0000,16688.7449
3,Итого,190000.0000,42670.1696,78318.0000,61000.6177


In [48]:
inf_factor_odin = (1 + dynamic['Ставка инфляции'])


# Делаем длинную 


In [50]:
# 1. Убираем строку 'Итого' из df_income
df_income_without_total = df_income[df_income['Год'] != 'Итого']

# 2. Теперь применяем melt к данным без итогов
df_long = df_income_without_total.melt(
    id_vars=['Год', 'Вложения'],
    value_vars=['ОФЗ ИН доход', 'ОФЗ ПД доход', 'Депозит доход'],
    var_name='Инструмент',
    value_name='Доход'
)

# 3. Сортируем по году и инструменту
df_long = df_long.sort_values(['Год', 'Инструмент']).reset_index(drop=True)

df_long

,Год,Вложения,Инструмент,Доход
0,2026,190000.0000,Депозит доход,26354.5517
1,2026,190000.0000,ОФЗ ИН доход,15656.0000
2,2026,190000.0000,ОФЗ ПД доход,26106.0000
3,2027,190000.0000,Депозит доход,17957.3211
4,2027,190000.0000,ОФЗ ИН доход,13242.2400
5,2027,190000.0000,ОФЗ ПД доход,26106.0000
6,2028,190000.0000,Депозит доход,16688.7449
7,2028,190000.0000,ОФЗ ИН доход,13771.9296
8,2028,190000.0000,ОФЗ ПД доход,26106.0000


In [44]:
ofz_pd

,Год,Привлекаемые средства,Количество человек,Ставка купона,На руках у человека,"Облигаций, штук",Купон,"Доход, руб",НДФЛ,Доход после вычета налога
0,2026,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
1,2027,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
2,2028,380000000000,2000000,0.1374,190000.0000,190.0000,137.4000,26106.0000,3393.7800,22712.2200
